# cPCA Analysis - Baseline Branch

Consumer notebook for pre-written baseline artifacts.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from contrastive import CPCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_mutual_info_score, silhouette_score, davies_bouldin_score

In [ ]:
ARTIFACT_ROOT = Path('/home/ben/MRSI_analysis/artifacts/processed_fft1000')
RESULTS_ROOT = Path('/home/ben/MRSI_analysis/artifacts/results_fft1000')
BRANCH = 'baseline'
N_COMPONENTS = 5
ALPHA_VALUE = 5
N_CLUSTERS = 3

In [ ]:
def load_branch_arrays(branch: str):
    branch_dir = ARTIFACT_ROOT / branch
    wmh = np.load(branch_dir / 'WMH.npy')
    back = np.load(branch_dir / 'BACK.npy')
    labels = np.load(branch_dir / 'Labels.npy')
    with open(branch_dir / 'metadata.json', 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    return wmh, back, labels, metadata


def run_cpca(wmh: np.ndarray, back: np.ndarray, labels: np.ndarray, n_components: int, alpha_value: float):
    mdl = CPCA(n_components=n_components)
    projected = mdl.fit_transform(
        wmh,
        back,
        plot=False,
        active_labels=labels,
        alpha_selection='manual',
        alpha_value=alpha_value
    )
    return mdl, projected


def cluster_projected(projected: np.ndarray, n_clusters: int):
    km = KMeans(n_clusters=n_clusters, random_state=42)
    clabels = km.fit_predict(projected)
    return clabels


def compute_metrics(labels: np.ndarray, cluster_labels: np.ndarray, projected: np.ndarray):
    return {
        'ami': float(adjusted_mutual_info_score(labels, cluster_labels)),
        'silhouette': float(silhouette_score(projected, cluster_labels)),
        'davies_bouldin': float(davies_bouldin_score(projected, cluster_labels))
    }


def contingency_table(labels: np.ndarray, cluster_labels: np.ndarray) -> pd.DataFrame:
    unique_clusters = np.unique(cluster_labels)
    unique_labels = np.unique(labels)
    matrix = np.zeros((len(unique_clusters), len(unique_labels)), dtype=int)
    for i, c in enumerate(unique_clusters):
        for j, s in enumerate(unique_labels):
            matrix[i, j] = int(np.sum((cluster_labels == c) & (labels == s)))
    return pd.DataFrame(
        matrix,
        index=[f'Cluster {int(c)}' for c in unique_clusters],
        columns=[f'Subject {int(s)}' for s in unique_labels]
    )


def plot_subject_scatter(projected: np.ndarray, labels: np.ndarray, title: str):
    colors = ['m', 'w', 'g', 'y']
    plt.figure(figsize=(6, 6))
    for i in range(1, 5):
        mask = labels == i
        plt.scatter(projected[mask, 0], projected[mask, 1], c=colors[i - 1], label=f'Sub {i}', s=40, alpha=0.8)
    plt.title(title)
    plt.xlabel('Contrastive PC1')
    plt.ylabel('Contrastive PC2')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
wmh, back, labels, branch_meta = load_branch_arrays(BRANCH)
mdl, projected = run_cpca(wmh, back, labels, N_COMPONENTS, ALPHA_VALUE)
cluster_labels = cluster_projected(projected, N_CLUSTERS)
metrics = compute_metrics(labels, cluster_labels, projected)
ctable = contingency_table(labels, cluster_labels)

print('Branch:', BRANCH)
print('WMH shape:', wmh.shape)
print('BACK shape:', back.shape)
print('Projected shape:', projected.shape)
print('Metrics:', metrics)
display(ctable)

plot_subject_scatter(projected, labels, f'cPCA Subject Scatter - {BRANCH}')

In [ ]:
out_dir = RESULTS_ROOT / BRANCH
out_dir.mkdir(parents=True, exist_ok=True)
np.save(out_dir / 'projected_data.npy', projected)
np.save(out_dir / 'cluster_labels.npy', cluster_labels)
ctable.to_csv(out_dir / 'contingency_table.csv')

with open(out_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

with open(out_dir / 'branch_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(branch_meta, f, indent=2)

print(f'Results written to: {out_dir}')